# 12 — Perceptroni dhe ura drejt rrjetave nervore

## Konteksti dhe problemi shkencor

Perceptroni mbyll trajtimin formal të kursit: një klasifikues linear online me teoremë konvergjence dhe kufizim të qartë në XOR.

### Objektivat e të nxënit

- të implementoni perceptronin;
- të kuptoni konvergjencën dhe margjinën;
- të demonstroni kufizimin XOR;
- të kuptoni në mënyrë përshkruese rrugën drejt MLP dhe deep learning.

> **Parimi i laboratorit:** matematikë → implementim nga parimet e para → implementim me bibliotekë → krahasim numerik → interpretim fizik.


In [ ]:
from pathlib import Path
import sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold, KFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso, Perceptron, SGDRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
DATA = Path("../datasets/synthetic")
SRC = Path("../src").resolve()
if str(SRC.parent) not in sys.path:
    sys.path.insert(0, str(SRC.parent))
np.set_printoptions(precision=4, suppress=True)
RANDOM_STATE = 42


## 1. Dataset linearisht i ndashëm

In [ ]:
df=pd.read_csv(DATA/'perceptron_separable.csv'); X=df[['x1','x2']].to_numpy(); y=df['class'].to_numpy(); plt.scatter(X[:,0],X[:,1],c=y,s=18); plt.show()

## 2. Rregulli i perceptronit

Me etiketa $y\in\{-1,+1\}$ përditësojmë vetëm kur $y(w^Tx+b)\le 0$:
$$w\leftarrow w+\eta yx,\qquad b\leftarrow b+\eta y.$$

In [ ]:
from src.perceptron.manual import fit_perceptron
w,b,mist=fit_perceptron(X,y,epochs=100,seed=0); print(w,b,mist); plt.plot(mist,marker='o'); plt.xlabel('epoka'); plt.ylabel('gabime'); plt.show()

## 3. Kufiri i vendimit

In [ ]:
xx=np.linspace(X[:,0].min()-.5,X[:,0].max()+.5,200); yy=-(w[0]*xx+b)/w[1]; plt.scatter(X[:,0],X[:,1],c=y,s=18); plt.plot(xx,yy); plt.show()

## 4. Krahasim me sklearn dhe regresion logjistik

In [ ]:
from sklearn.linear_model import Perceptron,LogisticRegression
sp=Perceptron(max_iter=1000,tol=1e-4,random_state=0).fit(X,y); lr=LogisticRegression().fit(X,y); print(sp.score(X,y),lr.score(X,y)); print('perceptron coef',sp.coef_,sp.intercept_); print('logistic',lr.coef_,lr.intercept_)

## 5. Margjina dhe numri i gabimeve

In [ ]:
# Afrojmë klasat duke zvogëluar distancën midis qendrave.
rng=np.random.default_rng(1); rows=[]
for sep in [2.0,1.2,.7,.4]:
 Xs=np.r_[rng.normal([-sep/2,-sep/2],.25,(100,2)),rng.normal([sep/2,sep/2],.25,(100,2))]; ys=np.r_[np.zeros(100,dtype=int),np.ones(100,dtype=int)]; _,_,mi=fit_perceptron(Xs,ys,epochs=100,seed=0); rows.append((sep,mi.sum(),mi[-1],len(mi)))
pd.DataFrame(rows,columns=['separation','total_updates','last_mistakes','epochs'])

## 6. XOR: kufiri i përfaqësimit

In [ ]:
dx=pd.read_csv(DATA/'xor.csv'); Xx=dx[['x1','x2']].to_numpy(); yx=dx['class'].to_numpy(); wx,bx,mx=fit_perceptron(Xx,yx,epochs=40,seed=0); print('mistakes per epoch=',mx); plt.scatter(Xx[:,0],Xx[:,1],c=yx,s=150); plt.show()

Rritja e epokave nuk e zgjidh XOR: problemi nuk është optimizimi por mungesa e një kufiri linear.

## 7. Një transformim veçorish

In [ ]:
Phi=np.c_[Xx,Xx[:,0]*Xx[:,1]]; wp,bp,mp=fit_perceptron(Phi,yx,epochs=100,seed=0); print(wp,bp,mp[-10:])

Ky transformim nuk është ende rrjet nervor; ai tregon se jolineariteti mund të futet në **përfaqësim**. Rrjetat me shtresa të fshehta mësojnë përfaqësime të tilla në mënyrë parametrike.

## 8. Teorema e konvergjencës: eksperiment numerik

Teorema jep kufirin $M\le(R/\gamma)^2$ kur të dhënat janë të ndashme me margjinë $\gamma$. Në notebook nuk provojmë vetëm formulën; ndryshojmë margjinën dhe shohim trendin e numrit të update-eve.

## 9. Çfarë vjen më tej — vetëm panoramë

Një shtresë e fshehtë llogarit $h=\phi(Wx+b)$. Kompozimi i shumë transformimeve jep MLP. Gradientët llogariten efikasisht me **prapapërhapje (backpropagation)**. CNN përdorin konvolucion; rrjetat rekurrente trajtojnë sekuenca; Transformer përdorin attention. Këto tema do të trajtohen rigorozisht në kursin pasues, jo këtu.

## 10. Ushtrime

- Implementoni perceptronin pa modul ndihmës.
- Provoni renditje të ndryshme të pikave dhe krahasoni numrin e update-eve.
- Krahasoni perceptronin me linear SVM.
- Shpjegoni me një skicë si dy njësi të fshehta mund të ndihmojnë XOR.

## 11. Shembull i plotë

Zgjidhni një dataset linearisht të ndashëm, trajnojeni manualisht, raportoni numrin e gabimeve, vizatoni kufirin dhe krahasoni me SVM dhe logistic regression.

## 12. Playground

Krijoni vetë dy klasa me margjina të ndryshme, outliers dhe zhurmë etiketash. Kur perceptroni nuk konvergon? Çfarë do të bënte SVM me margjinë të butë?